In [1]:
import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm
import timm

# =====================================================
# 1️⃣ CẤU HÌNH & TỐI ƯU HÓA HỆ THỐNG
# =====================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True # Tăng tốc cho input cố định size

set_seed(42)

DATA_DIR = "/kaggle/input/datasets/chanhhieudo/fn-ds-split/medical-split-fn"
BATCH_SIZE = 32        # Tăng lên 32 để tận dụng GPU, giảm thời gian train
NUM_EPOCHS = 60
PATIENCE = 10          # Dừng sớm nếu loss không giảm sau 10 epoch
LR_MAX = 5e-5          # Swin-B hoạt động tốt với LR thấp
ALPHA_MIX = 0.4  
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 2️⃣ DATA LOADER & AUGMENTATION
# =====================================================
def safe_loader(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), (0, 0, 0))

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

image_datasets = {
    "train": datasets.DatasetFolder(os.path.join(DATA_DIR, "train"), loader=safe_loader, extensions=("jpg","png","jpeg"), transform=train_transforms),
    "val": datasets.DatasetFolder(os.path.join(DATA_DIR, "val"), loader=safe_loader, extensions=("jpg","png","jpeg"), transform=val_transforms),
    "test": datasets.DatasetFolder(os.path.join(DATA_DIR, "test"), loader=safe_loader, extensions=("jpg","png","jpeg"), transform=val_transforms)
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == "train"),
                  num_workers=4, pin_memory=True, prefetch_factor=2, persistent_workers=True)
    for x in ["train", "val", "test"]
}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val", "test"]}
num_classes = len(image_datasets["train"].classes)

# =====================================================
# 3️⃣ MÔ HÌNH, LOSS & OPTIMIZER
# =====================================================
model = timm.create_model('swin_base_patch4_window7_224.ms_in22k_ft_in1k', pretrained=True, num_classes=num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=0.05)

# OneCycleLR: Tự động xử lý Warmup và giảm LR theo hình Cosine cực nhanh
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_MAX, 
    steps_per_epoch=len(dataloaders["train"]), 
    epochs=NUM_EPOCHS, pct_start=0.1
)
scaler = torch.amp.GradScaler()

# =====================================================
# 4️⃣ UTILS: MIXUP & EARLY STOPPING
# =====================================================
def mixup_data(x, y, alpha=ALPHA_MIX):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    return mixed_x, y, y[index], lam

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# =====================================================
# 5️⃣ TRAINING LOOP
# =====================================================
best_acc = 0.0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
early_stopper = EarlyStopping(patience=PATIENCE)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    for phase in ["train", "val"]:
        model.train(phase == "train")
        running_loss, running_corrects = 0.0, 0.0
        pbar = tqdm(dataloaders[phase], desc=f"{phase.upper()}", ncols=100)

        for inputs, labels in pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == "train"):
                with torch.amp.autocast(device_type="cuda"):
                    if phase == "train":
                        inputs, y_a, y_b, lam = mixup_data(inputs, labels)
                        outputs = model(inputs)
                        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
                    else:
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)

                if phase == "train":
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step() # Cập nhật LR theo từng bước (Batch-wise)

            _, preds = outputs.max(1)
            running_loss += loss.item() * inputs.size(0)
            if phase == "train":
                running_corrects += (lam * preds.eq(y_a).sum().item() + (1 - lam) * preds.eq(y_b).sum().item())
            else:
                running_corrects += preds.eq(labels).sum().item()
            
            pbar.set_postfix({"L": f"{loss.item():.3f}"})

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects / dataset_sizes[phase]
        history[f"{phase}_loss"].append(epoch_loss)
        history[f"{phase}_acc"].append(epoch_acc)
        
        print(f"{phase.upper()} - Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        if phase == "val":
            if epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), "best_swin_base_model.pth")
                print("⭐ Model Saved!")
            
            early_stopper(epoch_loss)

    if early_stopper.early_stop:
        print("🛑 Early stopping triggered!")
        break

# =====================================================
# 6️⃣ EVALUATION
# =====================================================
model.load_state_dict(torch.load("best_swin_base_model.pth"))
model.eval()
test_corrects = 0

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders["test"], desc="TESTING"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(inputs)
            test_corrects += outputs.max(1)[1].eq(labels).sum().item()

print(f"\n✅ Final Test Accuracy: {test_corrects / dataset_sizes['test']:.4f}")

# Vẽ đồ thị
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(history["train_loss"], label="Train"); plt.plot(history["val_loss"], label="Val"); plt.title("Loss"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history["train_acc"], label="Train"); plt.plot(history["val_acc"], label="Val"); plt.title("Accuracy"); plt.legend()
plt.savefig("history.png"); plt.show()


Epoch 1/60


TRAIN: 100%|█████████████████████████████████████████████| 445/445 [03:29<00:00,  2.12it/s, L=5.257]


TRAIN - Loss: 5.3108 Acc: 0.0127


VAL: 100%|███████████████████████████████████████████████| 126/126 [00:21<00:00,  5.99it/s, L=4.812]


VAL - Loss: 5.0952 Acc: 0.0507
⭐ Model Saved!

Epoch 2/60


...

TRAIN - Loss: 1.4526 Acc: 0.8180


VAL: 100%|███████████████████████████████████████████████| 126/126 [00:20<00:00,  6.06it/s, L=0.967]


VAL - Loss: 1.1581 Acc: 0.9189

Epoch 53/60


TRAIN:  71%|████████████████████████████████▏            | 318/445 [02:28<00:59,  2.15it/s, L=1.485]


KeyboardInterrupt: 